# Structured credit 1 — CLO debt tranches (AAA & BB)

**Prerequisites:** **02 / instruments** (bonds, credit derivatives).

A **CLO** (collateralized loan obligation) repackages a pool of floating-rate
leveraged loans into a senior/subordinate stack. This notebook prices the
**AAA senior** and **BB mezzanine** *debt* tranches and computes the metrics a
desk quotes them on. The **equity / residual** tranche gets its own notebook
(*structured credit 2*); consumer **ABS & RMBS** are in *structured credit 3*.

## Concept

- **Collateral:** first-lien leveraged loans paying **SOFR + spread**, so the
  asset *and* the liabilities float — credit spread, not rates, is the main risk.
- **Capital structure:** **AAA** (0–80%, SOFR+150), **BB** (80–92%, SOFR+600),
  **equity** (92–100%). Lower tranches are *subordinated* — they absorb losses first.
- **Reinvestment period:** early in life the manager **reinvests** principal into
  new loans instead of paying down notes, holding the pool balance flat. CLOs are
  often modeled with a sub-par **reinvestment price** (e.g. **97**): $1 of principal
  buys `1 / 0.97` of par, *building par* over time.
- **Debt metrics:** price (% of par), **WAL**, **discount margin**, **OAS**, and
  **break-even CDR** (how much collateral default the tranche absorbs loss-free).

In [ ]:
import json
import pandas as pd
from datetime import date

from finstack_quant.core.market_data import DiscountCurve, ForwardCurve, MarketContext
from finstack_quant.valuations.instruments.fixed_income import StructuredCredit

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
AS_OF = "2024-06-15"
AS_OF_D = date(2024, 6, 15)
print("Imports OK (structured credit).")

In [ ]:
# --- thin builders over the structured-credit serde spec (Rust is canonical) ---
def money(x):
    return {"amount": str(x), "currency": "USD"}

def asset(aid, atype, bal, rate, spread_bps=None, index_id=None,
          maturity="2031-06-15", day_count="Act360"):
    """One pool asset. Floating loans set spread_bps + index_id (rate = fallback)."""
    return {"id": aid, "asset_type": atype, "balance": money(bal), "rate": rate,
            "spread_bps": spread_bps, "index_id": index_id, "maturity": maturity,
            "credit_quality": None, "industry": None, "obligor_id": None,
            "is_defaulted": False, "recovery_amount": None, "purchase_price": None,
            "acquisition_date": None, "day_count": day_count,
            "smm_override": None, "mdr_override": None}

def floating(spread_bp, index_id="USD-SOFR-3M"):
    """Minimal floating tranche coupon (index + spread); other fields take engine defaults."""
    return {"Floating": {"index_id": index_id, "spread_bp": spread_bp,
            "reset_freq": {"count": 3, "unit": "months"}, "dc": "Act360", "calendar_id": "nyse"}}

def fixed(rate):
    return {"Fixed": {"rate": rate}}

def tranche(tid, att, det, seniority, bal, coupon, priority, freq_n=3, maturity="2031-06-15"):
    """One liability tranche. attachment/detachment are 0-100 (percent of structure)."""
    return {"id": tid, "attachment_point": att, "detachment_point": det,
            "behavior_type": "Standard", "seniority": seniority, "rating": None,
            "original_balance": money(bal), "current_balance": money(bal), "target_balance": None,
            "coupon": coupon, "oc_trigger": None, "ic_trigger": None,
            "credit_enhancement": {"subordination": money(0), "overcollateralization": money(0),
                                   "reserve_account": money(0), "excess_spread": 0.0,
                                   "cash_trap_active": False},
            "frequency": {"count": freq_n, "unit": "months"}, "day_count": "Act360",
            "deferred_interest": money(0), "pik_enabled": False, "is_revolving": False,
            "can_reinvest": False, "maturity": maturity, "expected_maturity": None,
            "payment_priority": priority, "attributes": {}}

def pool(pid, deal_type, assets, reinvest_end=None):
    """Collateral pool. reinvest_end (a date string) opens a reinvestment period."""
    rp = None
    if reinvest_end is not None:
        rp = {"end_date": reinvest_end, "is_active": True,
              "criteria": {"max_price": 101.0, "min_yield": 0.0,
                           "maintain_credit_quality": True, "maintain_wal": True,
                           "apply_eligibility_criteria": True}}
    return {"id": pid, "deal_type": deal_type, "base_currency": "USD", "assets": assets,
            "cumulative_defaults": money(0), "cumulative_recoveries": money(0),
            "cumulative_prepayments": money(0), "cumulative_scheduled_amortization": None,
            "reinvestment_period": rp, "collection_account": money(0),
            "reserve_account": money(0), "excess_spread_account": money(0), "rep_lines": None}

def build(did, deal_type, pool_dict, tranches, freq_n, closing, first_pay, maturity,
          reinvest_end=None, prepay=None, default=None, recovery=None, reinvestment_price=None):
    """Assemble + return (StructuredCredit, spec). reinvestment_price is % of par (e.g. 97)."""
    prepay = prepay if prepay is not None else {"cpr": 0.0, "curve": None}
    default = default if default is not None else {"cdr": 0.0, "curve": None}
    recovery = recovery if recovery is not None else {"rate": 0.40, "recovery_lag": 0}
    total = sum(int(t["original_balance"]["amount"]) for t in tranches)
    spec = {"id": did, "deal_type": deal_type, "pool": pool_dict,
            "tranches": {"tranches": tranches, "total_size": money(total)},
            "closing_date": closing, "first_payment_date": first_pay,
            "reinvestment_end_date": reinvest_end, "maturity": maturity,
            "frequency": {"count": freq_n, "unit": "months"},
            "payment_calendar_id": "nyse", "discount_curve_id": "USD-OIS",
            "pricing_overrides": {"mc_paths": 1}, "attributes": {},
            "prepayment_spec": prepay, "default_spec": default, "recovery_spec": recovery,
            "stochastic_prepay_spec": {"model": "Deterministic", **prepay},
            "stochastic_default_spec": {"model": "Deterministic", **default},
            "market_conditions": {"refi_rate": 0.04, "original_rate": None, "hpa": None,
                                  "unemployment": None, "seasonal_factor": 1.0, "custom_factors": {}},
            "credit_factors": {"credit_score": None, "dti": None, "ltv": None,
                               "delinquency_days": 0, "unemployment_rate": None, "custom_factors": {}}}
    if reinvestment_price is not None:
        spec["behavior_overrides"] = {"reinvestment_price": reinvestment_price}
    d = StructuredCredit(spec=spec)
    d.validate()
    return d, spec

## Market context

In [ ]:
# OIS discount curve (PV) + 3M-SOFR forward curve (projects floating coupons).
ois = DiscountCurve("USD-OIS", AS_OF_D,
                    [(0.0, 1.0), (1.0, 0.95), (3.0, 0.86), (5.0, 0.78), (8.0, 0.66), (10.0, 0.60)],
                    day_count="act_365f")
sofr = ForwardCurve("USD-SOFR-3M", 0.25,
                    [(0.0, 0.053), (3.0, 0.045), (8.0, 0.040)],
                    base_date=AS_OF_D, day_count="act_360")
market_json = MarketContext().insert(ois).insert(sofr).to_json()
print("Market:", [c.get("id") for c in json.loads(market_json)["curves"]])

In [ ]:
def tranche_results(deal):
    """Per-tranche Monte-Carlo results (npv, average_life, credit_duration, expected_loss)."""
    out = json.loads(deal.price(market_json, AS_OF, "structured_credit_stochastic"))
    return out["details"]["data"]["tranche_results"]

def price_table(deal, spec):
    """Tidy DataFrame of PV, price (% of par), WAL and expected loss per tranche."""
    orig = {t["id"]: int(t["original_balance"]["amount"]) for t in spec["tranches"]["tranches"]}
    rows = []
    for t in tranche_results(deal):
        ob = orig[t["tranche_id"]]
        pv = float(t["npv"]["amount"])
        rows.append({"tranche": t["tranche_id"], "seniority": t["seniority"],
                     "orig_$mm": ob / 1e6, "PV_$mm": pv / 1e6,
                     "price_%par": 100 * pv / ob if ob else float("nan"),
                     "WAL_yrs": t["average_life"], "credit_dur": t["credit_duration"],
                     "exp_loss_$mm": float(t["expected_loss"]["amount"]) / 1e6})
    return pd.DataFrame(rows)

## Build the CLO

A \$500mm pool of two first-lien loans, a \$400mm AAA / \$60mm BB / \$40mm equity
stack, quarterly pay, 3.5-year reinvestment period, ~7-year final. Behavioral
assumptions: **2% CDR**, **60% recovery** (40% severity), no prepayment.

In [ ]:
def build_clo(reinvest_end="2027-06-15", reinvestment_price=None,
              cdr=0.02, cpr=0.0, severity=0.40, recovery_lag=3):
    assets = [
        asset("LOAN_A", {"type": "FirstLienLoan", "industry": "Technology"},
              250_000_000, 0.0575, spread_bps=450.0, index_id="USD-SOFR-3M"),
        asset("LOAN_B", {"type": "FirstLienLoan", "industry": "Healthcare"},
              250_000_000, 0.0525, spread_bps=425.0, index_id="USD-SOFR-3M"),
    ]
    tranches = [
        tranche("CLASS-A", 0.0, 80.0, "Senior", 400_000_000, floating(150.0), 1),
        tranche("CLASS-B", 80.0, 92.0, "Mezzanine", 60_000_000, floating(600.0), 2),
        tranche("EQUITY", 92.0, 100.0, "Equity", 40_000_000, fixed(0.0), 3),
    ]
    return build("CLO-DEMO", "CLO", pool("POOL-CLO", "CLO", assets, reinvest_end),
                 tranches, 3, "2024-06-15", "2024-09-15", "2031-06-15",
                 reinvest_end=reinvest_end, reinvestment_price=reinvestment_price,
                 prepay={"cpr": cpr, "curve": None},
                 default={"cdr": cdr, "curve": None},
                 recovery={"rate": 1.0 - severity, "recovery_lag": recovery_lag})

clo, clo_spec = build_clo()
price_table(clo, clo_spec)

## With vs without reinvestment, and a 97 reinvestment price

Turning the reinvestment period **off** lets principal amortize the notes from
day one (shorter WAL). With it **on**, the pool stays full and the notes ride
longer. A **97 reinvestment price** builds extra par — most of that value accrues
to the residual, but it also thickens the cushion under the debt.

In [ ]:
rows = []
for label, reinv, px in [("no reinvestment", None, None),
                         ("reinvest @ par (100)", "2027-06-15", 100.0),
                         ("reinvest @ 97", "2027-06-15", 97.0)]:
    d, s = build_clo(reinvest_end=reinv, reinvestment_price=px, cpr=0.20)
    pt = price_table(d, s).set_index("tranche")
    for tr in ("CLASS-A", "CLASS-B"):
        rows.append({"scenario": label, "tranche": tr,
                     "price_%par": pt.loc[tr, "price_%par"], "WAL_yrs": pt.loc[tr, "WAL_yrs"]})
reinv_df = pd.DataFrame(rows).pivot(index="scenario", columns="tranche",
                                    values=["price_%par", "WAL_yrs"])
reinv_df

## Debt-tranche metrics: z-spread, CS01, discount margin, OAS, break-even CDR

These are computed **per tranche** from each note's own cashflows (the deal-level
metric registry aggregates all tranches into one stream, which isn't meaningful
per note — so we use the per-tranche API):

- **Z-spread / CS01 / spread duration** — `tranche_metrics(...)` returns the credit
  z-spread to a quoted price, its 1 bp DV01, and the spread duration, plus PV/WAL.
- **Discount margin (DM)** — full-reprice margin matching a target price. At the
  tranche's own PV it returns ~the quoted margin; at a *cheaper* price it widens.
- **OAS** — Monte-Carlo option-adjusted spread to a quoted market price.
- **Break-even CDR** — highest constant default rate the tranche takes *no* writedown at.

In [ ]:
orig = {t["id"]: int(t["original_balance"]["amount"]) for t in clo_spec["tranches"]["tranches"]}
res = {t["tranche_id"]: t for t in tranche_results(clo)}

rows = []
for tr, mkt_px in [("CLASS-A", 99.5), ("CLASS-B", 98.0)]:
    pv = float(res[tr]["npv"]["amount"])
    tm = json.loads(clo.tranche_metrics(market_json, AS_OF, tr, mkt_px))     # per-note z-spread/CS01
    dm_quoted = clo.discount_margin(market_json, AS_OF, tr, pv)              # ~ quoted margin
    dm_cheap = clo.discount_margin(market_json, AS_OF, tr, mkt_px / 100 * orig[tr])
    oas = json.loads(clo.oas(market_json, AS_OF, tr, mkt_px))
    becdr = clo.breakeven_cdr(market_json, AS_OF, tr)
    rows.append({"tranche": tr, "mkt_px": mkt_px,
                 "z_spread_bp": tm["z_spread_bp"], "CS01_$": tm["cs01"],
                 "spread_dur": tm["spread_duration"],
                 "DM@PV_bp": dm_quoted * 1e4, "DM@mkt_px_bp": dm_cheap * 1e4,
                 "OAS@mkt_px_bp": oas["oas"] * 1e4, "breakeven_CDR_%": becdr * 100})
pd.DataFrame(rows)

## Scenario / stress table (CDR × severity)

The desk's standard stress: reprice the tranche across a grid of default and
severity assumptions. BB sits just above equity, so it is the first *debt* class
to take writedowns as CDR rises.

In [ ]:
grid = json.dumps({"cprs": [0.0], "cdrs": [0.01, 0.03, 0.05, 0.08],
                   "severities": [0.30, 0.45, 0.60], "recovery_lag": 3})
cells = json.loads(clo.scenario_table(market_json, AS_OF, "CLASS-B", grid))["cells"]
sc = pd.DataFrame(cells)
display(sc.pivot(index="cdr", columns="severity", values="price").rename_axis(
    index="CDR", columns="severity").style.format("{:.1f}").set_caption("CLASS-B price (% par)"))
sc.pivot(index="cdr", columns="severity", values="writedown").rename_axis(
    index="CDR", columns="severity")

## Takeaways

- CLO debt is a **floating-rate, spread-risk** instrument: the AAA prices near par
  with a tight DM; the BB carries a much wider margin and a lower break-even CDR.
- **Reinvestment** lengthens WAL and keeps the pool working; a **97 reinvestment
  price** builds par that thickens subordination under the debt.
- Mark and stress the notes with **per-tranche z-spread / CS01, OAS, discount
  margin, break-even CDR** and the **CDR×severity** grid — all computed from each
  note's *own* cashflows. Equity economics are explored in *structured credit 2*.